(dkist:examples:vbi-extents)=

# Showing the Field of View of VBI on AIA


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.time import Time

import sunpy.map
from sunpy.net import Fido, attrs as a
from sunpy.visualization import drawing

import dkist
import dkist.net
from dkist.data.sample import VBI_L1_NZJTB

```{note}
This example requires `sunpy>=6.1` for the {obj}`sunpy.visualization.extent` function.
```


## Obtaining some data

In this example we will use the VBI sample dataset [AJQWW](https://dkist.data.nso.edu/datasetview/AJQWW).
If you want to replace this dataset with your own dataset, see <a href="https://docs.dkist.nso.edu/projects/python-tools/en/stable/howto_guides/sample_data.html#dkist-howto-guide-sample-data" target="_blank" style="text-decoration: underline">Downloading the Sample Data with Globus</a>.

Let's load the data with <a href="https://docs.dkist.nso.edu/projects/python-tools/en/stable/api/dkist.load_dataset.html#dkist.load_dataset" target="_blank" style="text-decoration: underline">`dkist.load_dataset`</a>:


In [ ]:
ds = dkist.load_dataset(VBI_L1_NZJTB)
ds

This gives us a <a href="https://docs.dkist.nso.edu/projects/python-tools/en/stable/api/dkist.TiledDataset.html#dkist.TiledDataset" target="_blank" style="text-decoration: underline">`dkist.TiledDataset`</a> object, which is an array of <a href="https://docs.dkist.nso.edu/projects/python-tools/en/stable/api/dkist.Dataset.html#dkist.Dataset" target="_blank" style="text-decoration: underline">`dkist.Dataset`</a> objects, as this VBI dataset is tiled in space (or mosaiced).

The sample data includes the ASDF file along with the FITS files for the first frame in each mosaic position.

First, let's extract the tiles which make up the first mosaic.


In [ ]:
first_tiles = ds.slice_tiles[0]

In [ ]:
fig = plt.figure(figsize=(12, 12))
fig = ds.plot(0, share_zscale=True, figure=fig)

Now let's extract the timestamps of each of these tiles


In [ ]:
times = Time([d.global_coords["time"] for d in first_tiles.flat]).sort()

And then download the AIA image closest to the time of the first tile within the range of all 9 tiles.


In [ ]:
results = Fido.search(a.Instrument.aia, a.Wavelength(171*u.AA), a.Time(times[0], times[-1], times[0]))
results

In [ ]:
aia_files = Fido.fetch(results, site="NSO")

Now we load the downloaded AIA file into a {obj}`sunpy.map.AIAMap` object.


In [ ]:
aia = sunpy.map.Map(aia_files)

In [ ]:
aia

Finally we can use a combination of <a href="https://docs.sunpy.org/en/stable/generated/api/sunpy.map.GenericMap.html#sunpy.map.GenericMap.plot" target="_blank" style="text-decoration: underline">`sunpy.map.GenericMap.plot`</a> and <a href="https://docs.sunpy.org/en/stable/generated/api/sunpy.visualization.drawing.extent.html#sunpy.visualization.drawing.extent" target="_blank" style="text-decoration: underline">`sunpy.visualization.drawing.extent`</a> to plot the extent of each tile on the disk.


In [ ]:
fig = plt.figure(figsize=(8,8))
ax = fig.add_subplot(projection=aia)

aia.plot(axes=ax)

# Iterate over each tile plotting the extent of the WCS for each one
for i, tile in enumerate(first_tiles.flat):
    drawing.extent(ax, tile.wcs, color=f"C{i}")

# Zoom in on the VBI region using pixel coordinates
_ = ax.axis((1500, 2000, 1900, 2400))